In [1]:
# handle the display
from IPython.core.display import HTML
display(HTML("<style>pre { white-space: pre !important; }</style>"))

In [2]:
# imports
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")


In [3]:
from pyspark import SparkConf, SparkContext

conf = SparkConf().setMaster("local").setAppName("RDD-Example")
sc = SparkContext(conf= conf)

24/01/09 15:19:20 WARN Utils: Your hostname, nalain-dell resolves to a loopback address: 127.0.1.1; using 10.200.120.6 instead (on interface wlp0s20f3)
24/01/09 15:19:20 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/01/09 15:19:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
# read the data 

path_to_data = Path("data/iris.csv")

iris = sc.textFile(str(path_to_data))
iris.collect()[:10]

['"sepal.length","sepal.width","petal.length","petal.width","variety"',
 '5.1,3.5,1.4,.2,"Setosa"',
 '4.9,3,1.4,.2,"Setosa"',
 '4.7,3.2,1.3,.2,"Setosa"',
 '4.6,3.1,1.5,.2,"Setosa"',
 '5,3.6,1.4,.2,"Setosa"',
 '5.4,3.9,1.7,.4,"Setosa"',
 '4.6,3.4,1.4,.3,"Setosa"',
 '5,3.4,1.5,.2,"Setosa"',
 '4.4,2.9,1.4,.2,"Setosa"']

In [7]:
# reemove the header 
iris_header = iris.first()
iris_header

('"sepal.length","sepal.width","petal.length","petal.width","variety"', str)

In [8]:
print(type(iris_header))

# convert header to rdd row 
iris_header = sc.parallelize([iris_header])
iris_header

<class 'str'>


ParallelCollectionRDD[5] at readRDDFromFile at PythonRDD.scala:289

In [9]:
# 
iris = iris.subtract(iris_header)
iris.collect()[:10]

['5.1,3.5,1.4,.2,"Setosa"',
 '4.9,3,1.4,.2,"Setosa"',
 '4.7,3.2,1.3,.2,"Setosa"',
 '5,3.6,1.4,.2,"Setosa"',
 '5.4,3.9,1.7,.4,"Setosa"',
 '4.6,3.4,1.4,.3,"Setosa"',
 '5,3.4,1.5,.2,"Setosa"',
 '5.4,3.7,1.5,.2,"Setosa"',
 '4.3,3,1.1,.1,"Setosa"',
 '5.1,3.5,1.4,.3,"Setosa"']

In [10]:
def parse_row(line:str)->tuple:
    cols = line.split(",")
    sepal_length = float(cols[0])
    species = str(cols[-1])
    return (species,sepal_length)
    

In [11]:
iris_parsed = iris.map(parse_row)
iris_parsed.collect()[:10]

[('"Setosa"', 5.1),
 ('"Setosa"', 4.9),
 ('"Setosa"', 4.7),
 ('"Setosa"', 5.0),
 ('"Setosa"', 5.4),
 ('"Setosa"', 4.6),
 ('"Setosa"', 5.0),
 ('"Setosa"', 5.4),
 ('"Setosa"', 4.3),
 ('"Setosa"', 5.1)]

In [14]:
# sum all sepal lengths 
iris_sl_totals = iris_parsed.mapValues(lambda x:(x,1)).reduceByKey(lambda x,y: (x[0]+y[0],x[1]+y[1]))
iris_sl_totals.collect()

[('"Setosa"', (250.3, 50)),
 ('"Versicolor"', (296.79999999999995, 50)),
 ('"Virginica"', (329.4, 50))]

In [16]:
# sepal length means per species 

iris_means = iris_sl_totals.mapValues(lambda x: x[0]/x[1])
results = iris_means.collect()

In [18]:
# do what you will with the results 

for result in results:
    print(f"{result[0]} == {result[1]}")

"Setosa" == 5.006
"Versicolor" == 5.935999999999999
"Virginica" == 6.587999999999999


In [19]:
sc.stop()